# Reproducing the 2022 baseline of Wdowiak with our pipeline

The published baseline of Wdowiak (25.1 BLEU en->scn, 29.1 scn->en) is a small Sockeye
Transformer trained on sentences he aligned by hand, scored on his own hand-selected test (the
121 trilingual lines from AS38-39, the `validation` sheet of his data) in the space of the
Napizia tokenizer. Here the same model, with his configuration, is trained on three sets that
differ only in where the parallel text comes from (built by `experiments/baseline/recipe_data.py`,
private, on Drive under `recipe/`):

- **A**: his hand-aligned sheets, which checks that our re-implementation reproduces his score;
- **B**: our automatic alignment of the born-digital *Arba Sicula* issues, plus the same textbook
  exercises (without Dieli's translations of Pitrè, which his sheet mixes with the journal, so B
  has less data than A);
- **C**: A and B together.

Each set is guarded against his test and against the 500 validation lines that stop the
training. The code is `experiments/baseline/recipe.py`; its docstring lists his configuration
and the two deliberate differences (we stop on our validation set, not on his test, and we train
from scratch). A T4 is enough; every model and every result is saved on Drive, so a
disconnected run resumes where it stopped. This notebook will become §7.12 of `sicilian_nmt.ipynb`.

In [ ]:
!pip -q install subword-nmt sacrebleu uv
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
!git -C SicilianNMT pull -q
# Sockeye 3 pins an old torch, so it gets its own environment
if not os.path.exists('/content/sk/bin/sockeye-train'):
    !uv venv -q --python 3.10 /content/sk && uv pip install -q --python /content/sk/bin/python 'sockeye>=3,<4' 'torch==1.13.1'
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'
sys.path.append('SicilianNMT/experiments/baseline')
import recipe
print('ready')

In [ ]:
# A (his data), B (ours), C (both), in both directions; skips what is already done
for name in ('A', 'B', 'C'):
    for s, t in (('en', 'scn'), ('scn', 'en')):
        recipe.run(name, f'{OUT}/recipe', s, t, sockeye_bin='/content/sk/bin')
res = json.load(open(f'{OUT}/recipe/results_recipe.json'))
print(f"{'set':6s} {'en->scn':>8s} {'scn->en':>8s}   (Wdowiak 2022: 25.1 / 29.1)")
for name in ('A', 'B', 'C'):
    print(f"{name:6s} {res.get(f'{name} en->scn', '-'):>8} {res.get(f'{name} scn->en', '-'):>8}")